# A01. 관측량 기준선 검토

**담당**: A · **상태**: 잠정안 (팀 동기화 때 확정) · **관련 문서**: `plan/01_데이터정의와_사례집계_확정.md` 1-2절

## 질문

지역×업종×집단 칸의 건수가 **몇 건 미만이면** 변화율을 믿지 못하고 "관측량 부족"으로 숨겨야 하는가?

## 잠정 결론

| 항목 | 잠정안 |
| --- | --- |
| 대상 | 지역×업종×집단 칸 |
| 기준 | `min(A기간 건수, B기간 건수) < 1,000` |
| 처리 | 수치 숨김, "관측량 부족" 표시, 실행 후보 없음 |
| 지역×업종 전체 | **같은 기준(1,000) 적용.** 현재 걸리는 곳 0곳. 기존 EDA의 지역 기준 5,000은 폐기 제안 (6절) |

기존 EDA의 "기간별 최소 1,000건"과 같은 값이다. 아래 셀이 그 근거다.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# 노트북 위치와 상관없이 프로젝트 루트를 찾는다
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "load.py").exists())
sys.path.insert(0, str(ROOT))

from src.load import BUZ_TARGETS, aggregate, load_card
from src.classify import CNT_BAND

pd.set_option("display.width", 160)
df = load_card()
print(f"{len(df):,}행, 지역 {df['region_key'].nunique()}개, CNT_BAND = ±{CNT_BAND}%p")

242,574행, 지역 255개, CNT_BAND = ±5.0%p


## 1. 칸 테이블 만들기

- 칸 = 지역(`region_key`) × 업종 × 집단(일반개인·외국인·법인). 255 × 2 × 3 = **1,530칸**
- 초과분 = 칸 변화율 − **같은 업종·같은 집단**의 전국 변화율
- `n_min` = A·B 기간 건수 중 **작은 쪽**

합계(A+B)가 아니라 작은 쪽을 쓰는 이유: 울릉군 서양음식 외국인은 16건 → 391건으로 합계는 407건이지만, 변화율의 분모는 16건이다. 분모가 작으면 변화율이 폭발한다.

In [2]:
frames = []
for buz in BUZ_TARGETS:
    nat = aggregate(df, buz=buz, group_by="group").set_index("group")
    r = aggregate(df, buz=buz, group_by=["region_key", "group"])
    r["buz"] = buz
    r["n_min"] = r[["cnt_A", "cnt_B"]].min(axis=1)
    r["cnt_ex"] = r["daily_cnt_chg_pct"] - r["group"].map(nat["daily_cnt_chg_pct"])
    r["unit_ex"] = r["unit_chg_pct"] - r["group"].map(nat["unit_chg_pct"])
    r["amt_ex"] = r["daily_amt_chg_pct"] - r["group"].map(nat["daily_amt_chg_pct"])
    frames.append(r)

cells = pd.concat(frames, ignore_index=True)
assert len(cells) == 1530, len(cells)
cells.head()

,region_key,group,amt_A,amt_B,cnt_A,cnt_B,daily_amt_A,daily_amt_B,daily_cnt_A,daily_cnt_B,...,unit_B,daily_amt_chg_pct,daily_cnt_chg_pct,unit_chg_pct,share_B_pct,buz,n_min,cnt_ex,unit_ex,amt_ex
0,강원특별자치도 강릉시,법인,3.993200e+08,4.862100e+08,9126.0,10839.0,4.436889e+06,5.342967e+06,101.400000,119.109890,...,44857.459175,20.421475,17.465375,2.516571,0.014664,8001,9126.0,-37.418391,7.988844,-25.986629
1,강원특별자치도 강릉시,외국인,5.924800e+08,6.787400e+08,14828.0,17034.0,6.583111e+06,7.458681e+06,164.755556,187.186813,...,39846.189973,13.300250,13.614872,-0.276920,0.020470,8001,14828.0,3.151710,2.003435,5.356039
2,강원특별자치도 강릉시,일반개인,1.527164e+10,1.624059e+10,366459.0,391100.0,1.696849e+08,1.784680e+08,4071.766667,4297.802198,...,41525.415495,5.176143,5.551289,-0.355415,0.489807,8001,366459.0,-1.784028,1.016310,-0.686828
3,강원특별자치도 고성군,법인,3.343000e+07,5.755000e+07,589.0,1124.0,3.714444e+05,6.324176e+05,6.544444,12.351648,...,51201.067616,70.258996,88.734864,-9.789324,0.001736,8001,589.0,33.851099,-4.317052,23.850892
4,강원특별자치도 고성군,외국인,5.276000e+07,6.737000e+07,1705.0,2091.0,5.862222e+05,7.403297e+05,18.944444,22.978022,...,32219.033955,26.288230,21.291612,4.119509,0.002032,8001,1705.0,10.828449,6.399864,18.344020


## 2. 이론 기준 — 우연에 의한 흔들림이 판정 밴드보다 작아야 한다

건수가 n건이면 우연만으로도 건수가 대략 √n건 흔들린다(포아송 근사). A→B 변화율의 표준오차는 약 **√(2/n)** 이다.

분류기(`src/classify.py`)는 건수 초과분이 **±5%p** 밖일 때 "방문 부진"으로 판정한다. 우연에 의한 흔들림이 이 밴드보다 크면 판정이 우연인지 실제인지 구분할 수 없다.

√(2/n) < 0.05 → **n > 800** → 올림해서 1,000

결과를 보고 고른 숫자가 아니라, 이미 정해 둔 밴드에서 유도되는 값이다. 단, 포아송 근사는 **하한**이다. 실제 흔들림은 계절·관광 요인이 더해져 이보다 크다 (3절).

In [3]:
band = CNT_BAND / 100
n_required = 2 / band**2
print(f"밴드 ±{CNT_BAND}%p를 넘지 않으려면 n > {n_required:.0f}")

pd.DataFrame({
    "건수": [250, 500, 800, 1000, 2000, 5000, 10000],
}).assign(
    우연에_의한_흔들림_pct=lambda t: (100 * np.sqrt(2 / t["건수"])).round(1),
    밴드보다_작음=lambda t: 100 * np.sqrt(2 / t["건수"]) < CNT_BAND,
)

밴드 ±5.0%p를 넘지 않으려면 n > 800


,건수,우연에_의한_흔들림_pct,밴드보다_작음
0,250,8.9,False
1,500,6.3,False
2,800,5.0,False
3,1000,4.5,True
4,2000,3.2,True
5,5000,2.0,True
6,10000,1.4,True


## 3. 실측 — 건수 구간별로 초과분이 얼마나 퍼지는가

건수가 충분히 크면 초과분의 퍼짐(IQR)이 안정돼야 한다. 작은 칸에서만 퍼짐이 커지면 그것이 잡음 구간이다.

- `cnt_IQR`, `unit_IQR`, `amt_IQR`: 각 초과분의 사분위 범위 (%p)
- `amt_abs_p90`: 금액 초과분 절댓값의 상위 10% — 화면에 뜰 극단값의 크기

In [4]:
bins = [0, 100, 250, 500, 1000, 2000, 5000, 10000, np.inf]
cells["구간"] = pd.cut(cells["n_min"], bins, right=False)

iqr = lambda s: s.quantile(0.75) - s.quantile(0.25)
by_bin = cells.groupby("구간", observed=True).agg(
    칸수=("n_min", "size"),
    cnt_IQR=("cnt_ex", iqr),
    unit_IQR=("unit_ex", iqr),
    amt_IQR=("amt_ex", iqr),
    amt_abs_p90=("amt_ex", lambda s: s.abs().quantile(0.9)),
).round(1)
by_bin

,칸수,cnt_IQR,unit_IQR,amt_IQR,amt_abs_p90
구간,,,,,
"[0.0, 100.0)",3,1079.9,17.0,1456.1,2636.2
"[100.0, 250.0)",6,55.7,28.1,120.1,174.8
"[250.0, 500.0)",17,40.8,14.8,36.5,58.6
"[500.0, 1000.0)",81,59.5,17.4,74.3,326.4
"[1000.0, 2000.0)",103,41.1,11.0,46.3,69.2
"[2000.0, 5000.0)",159,26.5,10.6,34.3,71.9
"[5000.0, 10000.0)",150,31.1,9.6,24.2,50.1
"[10000.0, inf)",1011,9.3,4.7,11.2,36.6


**읽는 법**

- 10,000건 이상 칸의 건수 초과분 IQR은 약 9%p로, 지역 전체 단위(한식 10.8%p)와 비슷하다. 여기가 "정상 퍼짐"이다.
- **500~1,000 구간**: 금액 초과분 상위 10%가 **300%p를 넘는다.** 기준을 500으로 두면 이 극단값이 화면에 그대로 뜬다.
- 1,000~2,000 구간도 여전히 퍼짐이 크다. 이 구간은 숨기지 않고, 규모 게이트(5%)와 우선순위 게이트에 맡긴다 (7절 한계 1).

## 4. 극단 사례 — 건수가 가장 작은 칸

In [5]:
cols = ["region_key", "buz", "group", "cnt_A", "cnt_B", "n_min",
        "daily_amt_chg_pct", "daily_cnt_chg_pct", "unit_chg_pct"]
cells.sort_values("n_min")[cols].head(10).round(1)

,region_key,buz,group,cnt_A,cnt_B,n_min,daily_amt_chg_pct,daily_cnt_chg_pct,unit_chg_pct
1072,경상북도 울릉군,8006,외국인,16.0,391.0,16.0,3064.8,2316.9,30.9
307,경상북도 울릉군,8001,외국인,61.0,520.0,61.0,974.3,743.1,27.4
306,경상북도 울릉군,8001,법인,71.0,210.0,71.0,187.9,192.5,-1.6
1060,경상북도 영양군,8006,외국인,104.0,200.0,104.0,162.7,90.2,38.1
1071,경상북도 울릉군,8006,법인,108.0,329.0,108.0,232.9,201.3,10.5
591,전라남도 신안군,8001,법인,155.0,296.0,155.0,43.7,88.9,-23.9
1356,전라남도 신안군,8006,법인,226.0,331.0,226.0,34.9,44.9,-6.9
15,강원특별자치도 양구군,8001,법인,229.0,327.0,229.0,73.3,41.2,22.7
603,전라남도 완도군,8001,법인,246.0,400.0,246.0,47.2,60.8,-8.5
612,전라남도 진도군,8001,법인,265.0,250.0,250.0,-15.1,-6.7,-9.0


울릉군 외국인 "금액 +3,065%" 같은 값은 실제 수요 변화가 아니라 **분모가 16건**이라 생긴 숫자다. 이런 칸이 외국인 규칙("외국인 금액 +1% 초과 → 다국어 메뉴")에 걸리면 엉터리 후보가 1순위로 뜬다.

## 5. 기준별 제외 규모

In [6]:
rows = []
total_amt_B = cells["amt_B"].sum()
for t in [300, 500, 1000, 2000]:
    ex = cells[cells["n_min"] < t]
    rows.append({
        "기준": t,
        "제외_칸": len(ex),
        "제외_비율_pct": round(len(ex) / len(cells) * 100, 1),
        "제외_B금액비중_pct": round(ex["amt_B"].sum() / total_amt_B * 100, 3),
        "일반개인": int((ex["group"] == "일반개인").sum()),
        "외국인": int((ex["group"] == "외국인").sum()),
        "법인": int((ex["group"] == "법인").sum()),
    })
pd.DataFrame(rows)

,기준,제외_칸,제외_비율_pct,제외_B금액비중_pct,일반개인,외국인,법인
0,300,11,0.7,0.003,0,4,7
1,500,26,1.7,0.014,0,7,19
2,1000,107,7.0,0.094,0,31,76
3,2000,210,13.7,0.221,0,77,133


1,000 기준이면 1,530칸 중 **107칸(7%)** 이 빠지지만, 금액으로는 **0.1%도 안 된다.** 빠지는 칸은 전부 작은 지역의 법인·외국인이고 일반개인은 한 칸도 빠지지 않는다.

In [7]:
cells.groupby(["buz", "group"])["n_min"].describe(percentiles=[0.05, 0.10]).round(0)

count      mean       std     min       5%      10%       50%        max
buz  group                                                                          
8001 법인     255.0   16804.0   35646.0    71.0    465.0    684.0    6559.0   277423.0
     외국인    255.0   17358.0   38429.0    61.0   1409.0   1958.0   11127.0   570202.0
     일반개인   255.0  286598.0  231885.0  5614.0  29518.0  39353.0  269433.0  1395578.0
8006 법인     255.0   24899.0   37251.0   108.0    589.0    762.0   12062.0   264399.0
     외국인    255.0   18846.0   35304.0    16.0    709.0   1128.0   11644.0   452628.0
     일반개인   255.0  271747.0  336243.0  3552.0  14807.0  19002.0  195879.0  3921250.0

## 6. 회귀 확인 — 4셀과 지역 단위는 영향이 없어야 한다

In [8]:
THRESHOLD = 1000

case_keys = {
    ("인천광역시 미추홀구", "8001"), ("경기도 이천시", "8006"),
    ("인천광역시 미추홀구", "8006"), ("경기도 이천시", "8001"),
}
case = cells[[(k, b) in case_keys for k, b in zip(cells["region_key"], cells["buz"])]]
assert len(case) == 12
assert (case["n_min"] >= THRESHOLD).all(), "4셀 집단 칸이 기준에 걸림"
print(f"4셀 12개 집단 칸 최소 건수: {case['n_min'].min():,.0f} (기준 {THRESHOLD:,})")

region = pd.concat(
    [aggregate(df, buz=b, group_by="region_key").assign(buz=b) for b in BUZ_TARGETS]
)
region_min = region[["cnt_A", "cnt_B"]].min(axis=1)
region["n_min"] = region_min
print(f"지역×업종 510칸 최소 건수: {region_min.min():,.0f} -> 1,000 기준 미달 {int((region_min < THRESHOLD).sum())}칸")
assert (region_min >= THRESHOLD).all()

assert int((cells["n_min"] < THRESHOLD).sum()) == 107
assert not ((cells["n_min"] < THRESHOLD) & (cells["group"] == "일반개인")).any()
print("회귀 확인 통과")

4셀 12개 집단 칸 최소 건수: 7,842 (기준 1,000)
지역×업종 510칸 최소 건수: 3,676 -> 1,000 기준 미달 0칸
회귀 확인 통과


### 기존 EDA의 지역 기준 5,000을 쓰지 않는 이유

기간별 건수 기준으로 5,000을 적용하면 아래 첫 행 **울릉군 서양음식**(A기간 3,676건)이 통째로 숨겨진다.

In [9]:
region.sort_values("n_min")[["region_key", "buz", "cnt_A", "cnt_B", "n_min",
                             "daily_cnt_chg_pct", "unit_chg_pct"]].head(5).round(1)

,region_key,buz,cnt_A,cnt_B,n_min,daily_cnt_chg_pct,unit_chg_pct
102,경상북도 울릉군,8006,3676.0,11915.0,3676.0,220.6,5.2
102,경상북도 울릉군,8001,5746.0,17478.0,5746.0,200.8,-7.3
98,경상북도 영양군,8006,5764.0,9257.0,5764.0,58.8,4.0
106,경상북도 청송군,8006,8849.0,10390.0,8849.0,16.1,-9.6
197,전라남도 신안군,8006,10291.0,11900.0,10291.0,14.4,5.4


울릉군 서양음식은 건수가 **+220%** 늘었다. 그러나 3,676건의 우연에 의한 흔들림은 √(2/3676) ≈ **±2.3%** 에 불과하다. 표본이 작아서 생긴 숫자가 아니라 **관광 계절성**(겨울 여객선 감축 → 봄 관광객 유입)이라는 실제 변화다.

건수 기준은 "표본이 작아 믿을 수 없는 칸"을 거르는 도구다. 실제로 일어난 계절 변화를 숨기는 데 쓰면 안 된다. 그래서 지역 단위에도 칸과 **같은 1,000 기준 하나만** 쓴다. 규칙이 하나라 C의 화면 구현도 단순해진다.

계절성이 큰 관광 지역을 어떻게 표시할지는 별도 문제로 남긴다 (7절 한계 4).

## 7. 결론과 한계

**잠정안**: `min(A기간 건수, B기간 건수) < 1,000`인 지역×업종×집단 칸은 "관측량 부족"으로 수치를 숨기고 실행 후보를 만들지 않는다. 구현 시 숫자를 코드에 박지 않고 `config/rules.yaml`의 설정값으로 둔다.

| 근거 | 내용 |
| --- | --- |
| 이론 | 우연에 의한 흔들림 √(2/n)이 `CNT_BAND` ±5%p보다 작아지는 지점 n > 800 |
| 실측 | 500~1,000 구간에 금액 초과분 300%p 이상의 극단값이 몰림 |
| 비용 | 107칸(7%) 제외, B기간 금액의 0.1% 미만, 일반개인 0칸 |
| 회귀 | 4셀 영향 없음, 기존 EDA 기간별 기준 1,000과 동일 |
| 단순화 | 칸·지역 모두 같은 기준 하나. 기존 지역 기준 5,000은 울릉군을 계절 변화 때문에 숨기므로 폐기 제안 |

**한계**

1. **1,000~2,000건 구간은 여전히 흔들림이 크다.** 작은 칸은 대부분 관광·농촌 지역이라 계절 변화가 실제로 크다. 이 구간은 규모 게이트(B기간 비중 5%)와 우선순위 게이트가 한 번 더 거른다.
2. **포아송 근사는 하한이다.** 카드 결제는 단체 결제처럼 한 번에 몰리는 경우가 있어 실제 흔들림은 더 크다.
3. **연령 테이블(`region_age_metrics`)은 칸이 더 잘게 쪼개진다.** 같은 기준을 쓰면 훨씬 많이 걸리므로, 만들 때 이 노트북의 5절을 연령 칸으로 다시 돌려 제외 규모를 확인한다.
4. **관광 계절성은 건수 기준으로 걸러지지 않는다.** 울릉군 서양음식 건수 +220%는 실제 변화지만 "지역 고유 문제"로 읽으면 안 된다. 계절성 경고 표시 여부는 팀 동기화 안건으로 올린다 (C의 유입형 상권 예외 처리와 함께 논의).